In [1]:
import pandas as pd

from src.features.nhanes_features import (
    create_basic_health_features,
    clean_features
)

ModuleNotFoundError: No module named 'src'

In [2]:
import os

os.getcwd()

'/workspaces/adaptive-health-engine/notebooks'

In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [5]:
import pandas as pd

from src.features.nhanes_features import (
    create_basic_health_features,
    clean_features
)

In [6]:
import pandas as pd

body = pd.read_sas(
    "../data/raw/nhanes/body.XPT"
)

demo = pd.read_sas(
    "../data/raw/nhanes/demographics.XPT"
)

print(body.shape)
print(demo.shape)

(8860, 22)
(11933, 27)


In [7]:
df = demo.merge(
    body,
    on="SEQN",
    how="inner"
)

print(df.shape)

(8860, 48)


In [8]:
df.columns.tolist()

['SEQN',
 'SDDSRVYR',
 'RIDSTATR',
 'RIAGENDR',
 'RIDAGEYR',
 'RIDAGEMN',
 'RIDRETH1',
 'RIDRETH3',
 'RIDEXMON',
 'RIDEXAGM',
 'DMQMILIZ',
 'DMDBORN4',
 'DMDYRUSR',
 'DMDEDUC2',
 'DMDMARTZ',
 'RIDEXPRG',
 'DMDHHSIZ',
 'DMDHRGND',
 'DMDHRAGZ',
 'DMDHREDZ',
 'DMDHRMAZ',
 'DMDHSEDZ',
 'WTINT2YR',
 'WTMEC2YR',
 'SDMVSTRA',
 'SDMVPSU',
 'INDFMPIR',
 'BMDSTATS',
 'BMXWT',
 'BMIWT',
 'BMXRECUM',
 'BMIRECUM',
 'BMXHEAD',
 'BMIHEAD',
 'BMXHT',
 'BMIHT',
 'BMXBMI',
 'BMDBMIC',
 'BMXLEG',
 'BMILEG',
 'BMXARML',
 'BMIARML',
 'BMXARMC',
 'BMIARMC',
 'BMXWAIST',
 'BMIWAIST',
 'BMXHIP',
 'BMIHIP']

In [9]:
health_features = create_basic_health_features(df)

health_features.head()

,age,weight_kg,height_cm,bmi
0,43.0,86.9,179.5,27.0
1,66.0,101.8,174.2,33.5
2,44.0,69.4,152.9,29.7
3,5.0,34.3,120.1,23.8
4,2.0,13.6,NaN,NaN


In [10]:
health_features["age"].describe()

count    8.860000e+03
mean     3.990542e+01
std      2.517546e+01
min      5.397605e-79
25%      1.500000e+01
50%      4.000000e+01
75%      6.300000e+01
max      8.000000e+01
Name: age, dtype: float64

In [11]:
adult_features = health_features[
    health_features["age"] >= 18
]

adult_features.shape


(6337, 4)

Raw NHANES participants:
8860

After adult filter (age ≥18):
6337 participants

Features:
4
- age
- weight_kg
- height_cm
- bmi

In [12]:
adult_features.isnull().sum()

age            0
weight_kg     89
height_cm     75
bmi          102
dtype: int64

adult_features
6337 participants

Missing values:

age          0
weight_kg   89
height_cm   75
bmi         102

In [13]:
clean_features = adult_features.dropna()

clean_features.shape

(6235, 4)

Adult NHANES dataset

Before cleaning:
6337 participants

After removing missing measurements:
6235 participants

Features:
- age
- weight_kg
- height_cm
- bmi

In [14]:
clean_features.to_csv(
    "../data/processed/nhanes_health_features.csv",
    index=False
)

In [15]:
import os

os.path.exists("../data/processed/nhanes_health_features.csv")

True

In [16]:
import pandas as pd

test_load = pd.read_csv(
    "../data/processed/nhanes_health_features.csv"
)

test_load.head()

,age,weight_kg,height_cm,bmi
0,43.0,86.9,179.5,27.0
1,66.0,101.8,174.2,33.5
2,44.0,69.4,152.9,29.7
3,34.0,90.6,173.3,30.2
4,68.0,103.5,155.9,42.6


In [17]:
from src.models.clustering import create_clusters

print("clustering module works")

clustering module works


In [18]:
import pandas as pd

health_data = pd.read_csv(
    "../data/processed/nhanes_health_features.csv"
)

health_data.shape

(6235, 4)

Processed dataset loaded:

6235 participants
4 features:

- age
- weight_kg
- height_cm
- bmi

In [19]:
X = health_data[
    [
        "age",
        "weight_kg",
        "height_cm",
        "bmi"
    ]
]

X.head()

,age,weight_kg,height_cm,bmi
0,43.0,86.9,179.5,27.0
1,66.0,101.8,174.2,33.5
2,44.0,69.4,152.9,29.7
3,34.0,90.6,173.3,30.2
4,68.0,103.5,155.9,42.6


X matrix:

6235 participants

Features:
- age
- weight_kg
- height_cm
- bmi

In [20]:
from src.models.clustering import create_clusters

labels, model = create_clusters(
    X,
    n_clusters=3
)

labels[:10]

array([0, 2, 0, 0, 2, 2, 1, 2, 2, 1], dtype=int32)

Input:
6235 participants
4 physiological features

Model:
KMeans

Clusters:
3

Output:
cluster labels generated ✅

In [21]:
clustered_data = X.copy()

clustered_data["cluster"] = labels

clustered_data.head()

,age,weight_kg,height_cm,bmi,cluster
0,43.0,86.9,179.5,27.0,0
1,66.0,101.8,174.2,33.5,2
2,44.0,69.4,152.9,29.7,0
3,34.0,90.6,173.3,30.2,0
4,68.0,103.5,155.9,42.6,2


In [22]:
cluster_summary = clustered_data.groupby("cluster").mean()

cluster_summary

,age,weight_kg,height_cm,bmi
cluster,,,,
0,32.569064,72.092121,167.247387,25.800776
1,66.811214,73.950319,164.195067,27.472818
2,50.424528,113.839084,171.877763,38.837332


KMeans produced 3 physiological profiles:

Cluster 0:
Age:        32.6 years
Weight:     72.1 kg
Height:     167.2 cm
BMI:        25.8

Cluster 1:
Age:        66.8 years
Weight:     73.9 kg
Height:     164.2 cm
BMI:        27.5

Cluster 2:
Age:        50.4 years
Weight:     113.8 kg
Height:     171.9 cm
BMI:        38.8

In [23]:
clustered_data["cluster"].value_counts()

cluster
1    2818
0    1933
2    1484
Name: count, dtype: int64

Cluster sizes:

Cluster 1:
2818 participants (45.2%)

Cluster 0:
1933 participants (31.0%)

Cluster 2:
1484 participants (23.8%)

Current interpretation:

Cluster 0
≈ younger/lower weight phenotype

Cluster 1
≈ older/moderate BMI phenotype

Cluster 2
≈ high adiposity phenotype

But there is a major limitation:

Right now the clustering is driven almost entirely by:

age
height
weight
BMI

It has not yet learned behavior.

This is only the physiological layer.

The future adaptive model needs:

Physiology
+
Executive function
+
ADHD traits
+
Lifestyle constraints
+
Adherence patterns

        ↓

Personalized intervention strategy

In [24]:
clustered_data.to_csv(
    "../data/processed/nhanes_health_clusters.csv",
    index=False
)

In [25]:
import os

os.path.exists("../data/processed/nhanes_health_clusters.csv")

True